# Modeling

Treina e compara vários modelos de classificação usando `data_preparation.py` e `data_modeling.py`, e guarda o melhor em `models/`.

In [1]:
from datetime import datetime
from pathlib import Path
import sys

SRC_PATH = Path("..").resolve() / "src"
sys.path.insert(0, str(SRC_PATH))

import data_preparation as dp
import data_modelling as dm


## 1. Preparar os dados

Reaproveita a fase anterior.

In [2]:
DATASET_NAME = "malware_memory_dump"
RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
DATA_PATH = Path("..").resolve() / "dataset" / "raw" / "MalwareMemoryDump.csv"
OUTPUT_DIR = Path("..").resolve() / "models"

X_train, X_val, X_test, y_train, y_val, y_test = dp.run_data_preparation(
    str(DATA_PATH),
    target="Label",
    random_state=42,
)


Dimensões do dataset: (58596, 58)

--- Valores nulos ---
Raw_Type                                  0
pslist_nproc                              0
pslist_nppid                              0
pslist_avg_threads                        0
pslist_nprocs64bit                        0
pslist_avg_handlers                       0
dlllist_ndlls                             0
dlllist_avg_dlls_per_proc                 0
handles_nhandles                          0
handles_avg_handles_per_proc              0
handles_nport                             0
handles_nfile                             0
handles_nevent                            0
handles_ndesktop                          0
handles_nkey                              0
handles_nthread                           0
handles_ndirectory                        0
handles_nsemaphore                        0
handles_ntimer                            0
handles_nsection                          0
handles_nmutant                           0
ldrmodules_not_in_l

## 2. Comparar modelos com validação cruzada

Testa Regressão Logística, kNN, SVM, Naive Bayes, MLP, Árvore de Decisão, Random Forest e Gradient Boosting, com `StratifiedKFold` (normalização aplicada só aos modelos que precisam).

In [3]:
results_df = dm.evaluate_models(X_train, y_train, scoring="f1_macro")
dm.save_results_csv(
    results_df,
    dataset_name=DATASET_NAME,
    output_dir=str(OUTPUT_DIR),
    timestamp=RUN_TIMESTAMP,
)
dm.plot_model_comparison(
    results_df,
    scoring="f1_macro",
    dataset_name=DATASET_NAME,
    output_dir=str(OUTPUT_DIR),
    timestamp=RUN_TIMESTAMP,
)


Regressão Logística    | f1_macro = 0.9989 ± 0.0003
kNN                    | f1_macro = 0.9998 ± 0.0001
SVM                    | f1_macro = 0.9996 ± 0.0002
Naive Bayes            | f1_macro = 0.9927 ± 0.0007
MLP                    | f1_macro = 0.9999 ± 0.0001
Árvore de Decisão      | f1_macro = 0.9997 ± 0.0002
Random Forest          | f1_macro = 0.9999 ± 0.0001
Gradient Boosting      | f1_macro = 0.9996 ± 0.0002
Comparação de modelos guardada em: C:\Users\Thiago\Desktop\IAAC\models\comparacao_modelos_malware_memory_dump_20261007_165451_257890.csv
Gráfico guardado em: C:\Users\Thiago\Desktop\IAAC\models\comparacao_modelos_malware_memory_dump_20261007_165451_257890.png


'C:\\Users\\Thiago\\Desktop\\IAAC\\models\\comparacao_modelos_malware_memory_dump_20261007_165451_257890.png'

In [4]:
dm.plot_model_comparison(results_df, scoring="f1_macro", dataset_name=DATASET_NAME, output_dir="../models", timestamp=RUN_TIMESTAMP)

Gráfico guardado em: C:\Users\Thiago\Desktop\IAAC\..\models\comparacao_modelos_malware_memory_dump_20261007_165451_257890.png


'C:\\Users\\Thiago\\Desktop\\IAAC\\..\\models\\comparacao_modelos_malware_memory_dump_20261007_165451_257890.png'

## 3. Escolher e treinar o modelo campeão

In [5]:
best_name = dm.select_best_model(results_df)
tuned_pipe = dm.train_final_model(
    best_name, X_train, y_train, X_val, y_val, scoring="f1_macro",
)
validation_metrics = dm.evaluate_on_validation(
    tuned_pipe, X_train, y_train, X_val, y_val, scoring="f1_macro",
)


Melhor modelo: Random Forest
Melhores hiperpar?metros para Random Forest: {'model__max_depth': None, 'model__n_estimators': 100}
f1_macro na valida??o: 1.0000

--- Treino vs valida??o (f1_macro) ---
Treino: 1.0000 | Valida??o: 1.0000 | Diferen?a: 0.0000


## 4. Avaliar no conjunto de teste

In [6]:
best_name = dm.select_best_model(results_df)
tuned_pipe = dm.train_final_model(
    best_name,
    X_train,
    y_train,
    X_val,
    y_val,
    scoring="f1_macro",
)
validation_metrics = dm.evaluate_on_validation(
    tuned_pipe,
    X_train,
    y_train,
    X_val,
    y_val,
    scoring="f1_macro",
)



Melhor modelo: Random Forest
Melhores hiperpar?metros para Random Forest: {'model__max_depth': None, 'model__n_estimators': 100}
f1_macro na valida??o: 1.0000

--- Treino vs valida??o (f1_macro) ---
Treino: 1.0000 | Valida??o: 1.0000 | Diferen?a: 0.0000


## 5. Guardar o modelo campe??o

O pipeline versionado e a c??pia `latest` ficam em `../models/`, acompanhados por metadados JSON com m??tricas e features. CSVs e gr??ficos de compara????o e avalia????o tamb??m s??o guardados nessa pasta com dataset e timestamp.

In [7]:
best_pipe, X_train_val, y_train_val = dm.refit_final_model(
    tuned_pipe,
    X_train,
    y_train,
    X_val,
    y_val,
)
test_metrics = dm.evaluate_on_test(
    best_pipe,
    X_train_val,
    y_train_val,
    X_test,
    y_test,
    scoring="f1_macro",
    dataset_name=DATASET_NAME,
    output_dir=str(OUTPUT_DIR),
    timestamp=RUN_TIMESTAMP,
)

model_path = dm.save_model(
    best_pipe,
    best_name,
    dataset_name=DATASET_NAME,
    output_dir=str(OUTPUT_DIR),
    metrics={
        "validation": validation_metrics,
        "test": test_metrics,
        "cv_score_medio": float(results_df.iloc[0]["media"]),
        "cv_score_desvio_padrao": float(results_df.iloc[0]["desvio_padrao"]),
    },
    feature_names=list(X_train.columns),
    timestamp=RUN_TIMESTAMP,
)
print(f"Modelo guardado em: {model_path}")



--- Classification report (teste) ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      4385
           1       1.00      1.00      1.00      4325

    accuracy                           1.00      8710
   macro avg       1.00      1.00      1.00      8710
weighted avg       1.00      1.00      1.00      8710


--- Overfitting check (f1_macro) ---
Treino: 1.0000 | Teste: 1.0000 | Diferença: 0.0000
Diferença treino-teste pequena — sem sinais fortes de overfitting.
Gráfico guardado em: C:\Users\Thiago\Desktop\IAAC\models\matriz_confusao_malware_memory_dump_20261007_165451_257890.png
Modelo guardado em: C:\Users\Thiago\Desktop\IAAC\models\malware_memory_dump_random_forest_20261007_165451_257890.joblib
Metadados guardados em: C:\Users\Thiago\Desktop\IAAC\models\malware_memory_dump_random_forest_20261007_165451_257890.json
Cópias 'latest' atualizadas em: C:\Users\Thiago\Desktop\IAAC\models\malware_memory_dump_random_forest_latest.joblib 

## Alternativa: correr tudo de uma vez

In [8]:
# best_pipe = dm.run_modeling(X_train, X_val, X_test, y_train, y_val, y_test, dataset_name=DATASET_NAME)